<a href="https://colab.research.google.com/github/R786P/telco_churn_predictor/blob/main/fooocus_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pygit2==1.15.1
%cd /content
!git clone https://github.com/lllyasviel/Fooocus.git
%cd /content/Fooocus
!python entry_with_update.py --share --always-high-vram


In [2]:
%%writefile /content/data_cleaning.py

import pandas as pd

def clean_churn_data(df: pd.DataFrame) -> pd.DataFrame:
    """Dummy function to simulate data cleaning."""
    print("\n--- Running Dummy Data Cleaning ---")
    # Add some dummy cleaning logic if needed, e.g., dropping NaNs
    # For now, just return the dataframe as is or with minimal modification
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df.dropna(inplace=True)
    print(f"DataFrame shape after dummy cleaning: {df.shape}")
    print("--- Dummy Data Cleaning Complete ---")
    return df

Writing /content/data_cleaning.py


In [3]:
%%writefile /content/feature_engineering.py

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def build_preprocessor(X_train: pd.DataFrame):
    """Dummy function to simulate feature engineering preprocessor."""
    print("\n--- Running Dummy Feature Engineering Preprocessor Building ---")

    # Identify categorical and numerical features
    categorical_features = X_train.select_dtypes(include=['object', 'category']).columns
    numerical_features = X_train.select_dtypes(include=['int64', 'float64']).columns

    # Create preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ],
        remainder='passthrough'
    )
    print("--- Dummy Feature Engineering Preprocessor Building Complete ---")
    return preprocessor, categorical_features, numerical_features

Writing /content/feature_engineering.py


I've created placeholder `data_cleaning.py` and `feature_engineering.py` files. You can now try running the main cell again. If you have specific cleaning or feature engineering logic, you can modify these files.

In [1]:
import sys
sys.path.append('/content/')

import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from data_cleaning import clean_churn_data
from feature_engineering import build_preprocessor
import os

def main():
    # Create dummy data and directories if they don't exist
    os.makedirs('data/raw', exist_ok=True)
    os.makedirs('models', exist_ok=True)
    if not os.path.exists('data/raw/churn.csv'):
        print('Creating dummy churn.csv')
        dummy_data = pd.DataFrame({
            'customerID': [f'C{i:04d}' for i in range(100)],
            'gender': ['Male', 'Female'] * 50,
            'SeniorCitizen': [0, 1] * 50,
            'Partner': ['Yes', 'No'] * 50,
            'Dependents': ['No', 'Yes'] * 50,
            'tenure': list(range(1, 101)),
            'PhoneService': ['Yes'] * 100,
            'MultipleLines': ['No'] * 100,
            'InternetService': ['DSL', 'Fiber optic'] * 50,
            'OnlineSecurity': ['No'] * 100,
            'OnlineBackup': ['Yes'] * 100,
            'DeviceProtection': ['No'] * 100,
            'TechSupport': ['Yes'] * 100,
            'StreamingTV': ['No'] * 100,
            'StreamingMovies': ['Yes'] * 100,
            'Contract': ['Month-to-month'] * 100,
            'PaperlessBilling': ['Yes'] * 100,
            'PaymentMethod': ['Electronic check'] * 100,
            'MonthlyCharges': [50.0 + i for i in range(100)],
            'TotalCharges': [100.0 + 50.0 * i for i in range(100)],
            'Churn': ['No', 'Yes'] * 50
        })
        dummy_data.to_csv('data/raw/churn.csv', index=False)

    # 1. Load & Clean
    df = clean_churn_data(pd.read_csv("data/raw/churn.csv"))
    X = df.drop("Churn", axis=1)
    y = df["Churn"]

    # 2. Stratified Split (important for imbalanced data)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. Preprocess
    preprocessor, _, _ = build_preprocessor(X_train)
    X_train_prep = preprocessor.fit_transform(X_train)
    X_test_prep = preprocessor.transform(X_test)

    # 4. Train Model
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",  # handles churn imbalance
        n_jobs=-1
    )
    model.fit(X_train_prep, y_train)

    # 5. Save Artifact
    artifact = {
        "model": model,
        "preprocessor": preprocessor,
        "feature_names": list(X.columns)
    }
    joblib.dump(artifact, "models/churn_rf_v1.pkl")
    print("✅ Model trained & saved to models/churn_rf_v1.pkl")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'data_cleaning'